# Statistical Arbitrage Strategy: Intraday Mean-Reversion Pairs Trading
##### Yevgen Revtsov

## Strategy Summary

### Introduction

This research project demonstrates that cointegrated equity pairs within the same economic sector can be systematically exploited for profit through mean-reversion trading, using the Engle-Granger two-step methodology to identify stationary spreads and z-score normalization to generate entry and exit signals. First, we apply cointegration testing to within-sector stock pairs to identify relationships with long-run equilibrium properties, filtering by statistical significance (p-value ≤ 0.05) and mean-reversion speed. Second, we construct dollar-neutral spread positions using estimated hedge ratios, implementing strict risk management through stop losses and time limits to control tail risk and improve capital efficiency. Third, we backtest the strategy on 1-minute intraday data from 100 large-cap US equities across 11 GICS sectors, incorporating realistic transaction costs to validate the profitability of high-frequency statistical arbitrage. Fourth, we conduct comprehensive hypothesis testing across critical dimensions including pair selection methodology, signal generation parameters, transaction cost sensitivity, and out-of-sample robustness to establish the strategy's validity and identify optimization opportunities. Fifth, we perform walk-forward analysis and parameter sensitivity tests to ensure the strategy is not overfit to historical data and maintains consistent risk-adjusted returns across different market regimes. The goal is to show how modern cointegration-based pairs trading can generate positive risk-adjusted returns in liquid equity markets through systematic exploitation of temporary mispricings, while recognizing the critical importance of transaction cost management, robust pair selection, and disciplined risk controls for strategy success.

### Point 1: Cointegration-Based Pair Selection

The strategy employs the Engle-Granger two-step cointegration methodology to identify stock pairs with stationary spread relationships suitable for mean-reversion trading. The first step estimates the hedge ratio $\beta$ through ordinary least squares regression:

$$Y_t = \alpha + \beta \cdot X_t + \epsilon_t$$

where $Y_t$ and $X_t$ represent the price series of two stocks at time $t$. The second step tests the residuals (spread $Z_t = Y_t - \beta \cdot X_t$) for stationarity using the Augmented Dickey-Fuller (ADF) test, with the null hypothesis that the spread contains a unit root. Pairs passing the stationarity test at 95% confidence (p-value ≤ 0.05) and exhibiting half-lives between 5 and 120 observations are selected, with stronger preference given to lower p-values indicating more robust cointegration relationships. The restriction to within-sector pairs (using GICS sector classification) increases the probability of finding economically meaningful cointegration, as stocks in the same sector share common factor exposures including industry trends, commodity prices, regulatory changes, and macroeconomic forces. The key takeaway is that systematic cointegration testing with appropriate filters provides a statistically rigorous foundation for identifying pairs with predictable mean-reversion behavior, superior to correlation-based or distance-based methods that lack theoretical grounding in long-run equilibrium relationships.

### Point 2: Mean-Reversion Signal Generation via Z-Score Normalization

The strategy generates trading signals by normalizing the spread through z-score transformation, allowing standardized comparison across pairs with different volatility characteristics. The z-score at time $t$ is calculated as:

$$z_t = \frac{Z_t - \mu_{rolling}}{\sigma_{rolling}}$$

where $Z_t$ is the current spread value, $\mu_{rolling}$ is the rolling mean over a lookback window (default 60 observations), and $\sigma_{rolling}$ is the rolling standard deviation over the same window. Entry signals are generated when the z-score exceeds threshold levels: long spread entry at $z < -2.5$ (spread undervalued, expect reversion upward) and short spread entry at $z > +2.5$ (spread overvalued, expect reversion downward). Exit signals occur at mean reversion ($|z| \leq 0.0$), stop loss ($|z| > 4.0$ indicating relationship breakdown), or time limit (to prevent capital tie-up and overnight risk).

### Point 3: Market-Neutral Position Construction and Risk Management

The strategy constructs dollar-neutral positions at entry by taking offsetting positions in both legs of each pair, with position sizes determined by the hedge ratio to minimize directional market exposure. For a long spread position (when $z < -2.5$), the strategy buys \$1 of stock A and sells $\beta$ of stock B; for a short spread position (when $z > +2.5$), it sells \$1 of stock A and buys $\beta$ of stock B, where $\beta$ is the cointegration coefficient estimated from the regression. Capital allocation follows an equal-weight scheme with a maximum of 10 simultaneous pairs, each receiving \$10,000 from a \$100,000 portfolio, ensuring diversification across uncorrelated pair relationships. Risk management operates on three levels: (1) stop losses at $|z| = 4.0$ limit catastrophic losses from cointegration breakdown, (2) time stops at 120 minutes prevent capital from being tied up in slow-reverting positions and eliminate overnight gap risk, and (3) position limits ensure no overlapping trades in the same pair and bound maximum portfolio exposure. Transaction costs are modeled conservatively at 20 basis points per leg, encompassing bid-ask spreads, commissions, and market impact, totaling 80 basis points per round-trip trade (4 legs: buy A, sell B, exit A, exit B). The critical takeaway is that disciplined risk management and realistic cost assumptions are essential for translating theoretical statistical arbitrage edges into actual trading profits, as high-frequency strategies are particularly sensitive to execution quality and cost drag.

### Point 4: Intraday Implementation with Market Hours Filtering

The strategy operates on 1-minute intraday data with careful filtering to exclude after-hours trading and handle market holidays, ensuring that only regular trading hours (9:30 AM - 4:00 PM ET) are analyzed when liquidity and price discovery are optimal. Data preprocessing uses pandas_market_calendars with NYSE schedule to generate valid trading minute timestamps, filtering out pre-market (before 9:30 AM) and post-market (after 4:00 PM) periods where spreads widen and execution quality deteriorates. Half-day holiday detection identifies early market closes (1:00 PM). All timestamps are converted to US/Eastern timezone for consistency, and the data pipeline handles corporate actions (splits, dividends) through EODHD API integration. The universe consists of 100 large-cap and mid-cap stocks stratified across 11 GICS sectors, selected for high liquidity and sector diversity. The intraday timeframe provides two key advantages over daily data: (1) more frequent mean-reversion opportunities increase trade frequency and capital velocity, potentially improving risk-adjusted returns, and (2) faster position turnover reduces overnight risk and exposure to gap events from news announcements. The tradeoff is higher transaction cost sensitivity, making realistic cost modeling and execution quality paramount for strategy viability.

### Point 5: Hypothesis Testing and Validation Framework

The project establishes a comprehensive hypothesis testing framework covering high-priority dimensions to validate strategy assumptions, optimize parameters, and assess robustness to overfitting and regime changes. Core validation hypotheses (Phase 1) test whether within-sector pairs exhibit higher cointegration rates than random pairs (H1), whether optimal z-score entry thresholds exist (H2), whether the strategy remains profitable under realistic transaction cost assumptions (H3), and whether in-sample optimization leads to significant out-of-sample degradation (H4). Robustness hypotheses (Phase 2) examine the effectiveness of stop losses in reducing tail risk (H5), walk-forward consistency across rolling windows (H6), and whether lower ADF p-values predict higher profitability (H7) and compare performance across different timeframes from 1-minute to daily (H8). For each hypothesis, the testing methodology specifies the null hypothesis (H₀), alternative hypothesis (H₁), test procedure, evaluation metrics, and expected outcomes. The key goal is that systematic hypothesis testing transforms pairs trading from a heuristic approach into a rigorous quantitative framework, enabling evidence-based parameter selection and providing early warning signals for strategy degradation.

### Conclusion

This statistical arbitrage strategy synthesizes cointegration theory, mean-reversion dynamics, and disciplined risk management to systematically exploit temporary mispricings in equity pairs while maintaining market neutrality and controlling downside risk. The Engle-Granger methodology provides a statistically rigorous framework for pair selection based on long-run equilibrium relationships rather than spurious correlations, z-score normalization enables standardized signal generation across heterogeneous pairs with adaptive threshold setting, market-neutral positioning combined with stop losses and time limits ensures that profitable mean-reversion opportunities are captured while limiting exposure to relationship breakdowns and overnight risk, intraday implementation captures high-frequency inefficiencies while requiring careful attention to transaction costs and execution quality, and comprehensive hypothesis testing establishes empirical validity while guarding against overfitting and parameter instability. The strategy is relevant to quantitative portfolio managers and systematic traders seeking market-neutral alpha sources with limited directional exposure, particularly those with access to intraday data feeds, low-latency execution infrastructure. Success hinges on continuous monitoring of pair relationships for structural breaks, realistic modeling of transaction costs and slippage, disciplined adherence to risk management rules especially during market stress, and ongoing research into regime-dependent parameters and machine learning enhancements.

---

## Hypothesis Testing Framework

This section presents the hypotheses identified for testing, organized by priority phase. Each hypothesis includes the formal null and alternative hypotheses, testing methodology, evaluation metrics, and expected outcomes.

### Phase 1: Core Validation

#### Hypothesis 1: Within-Sector Cointegration

**Null Hypothesis (H₀)**: Stock pairs within the same economic sector are no more likely to be cointegrated than random pairs across sectors.

**Alternative Hypothesis (H₁)**: Within-sector pairs exhibit significantly higher cointegration rates than cross-sector pairs.

**Test Method**:
- Compare cointegration test pass rates (p < 0.05) for:
  - Within-sector pairs (stocks in same GICS sector)
  - Cross-sector random pairs (random pairing across different sectors)
- Use chi-square test for independence to determine if sector membership affects cointegration probability
- Calculate odds ratio to quantify the strength of the relationship
- Sample size: test all possible within-sector pairs vs. 1000 random cross-sector pairs

**Metrics**:
- Percentage of pairs passing ADF test at p < 0.05 (within-sector vs. cross-sector)
- Mean ADF statistic by pair type (more negative = stronger stationarity evidence)
- Mean p-value by pair type (lower = stronger cointegration)
- Odds ratio and 95% confidence interval
- Chi-square test statistic and p-value for independence test

**Expected Outcome**: Confirm H₁ (within-sector pairs significantly more cointegrated)

**Implications**:
- Validates sector-stratified universe design
- Justifies limiting computational search to within-sector pairs only
- Provides economic rationale (shared factor exposures) for statistical relationships
- If H₀ cannot be rejected: need to reconsider sector restriction or explore alternative grouping schemes (e.g., industry groups, factor-based clustering)

#### Hypothesis 2: Z-Score Entry Threshold Optimization

**Null Hypothesis (H₀)**: The choice of z-score entry threshold (e.g., 2.0 vs. 2.5 vs. 3.0) does not significantly affect risk-adjusted returns.

**Alternative Hypothesis (H₁)**: There exists an optimal z-score entry threshold that maximizes Sharpe ratio, balancing trade frequency and signal quality.

**Test Method**:
- Grid search over z_entry ∈ {1.5, 2.0, 2.5, 3.0, 3.5}
- For each threshold, run complete backtest on same data
- Calculate Sharpe ratio for each threshold
- Analyze tradeoff curves: Sharpe vs. threshold, frequency vs. threshold, win rate vs. threshold
- Perform sensitivity analysis: does optimal threshold vary by sector or pair characteristics?

**Metrics**:
- Sharpe ratio vs. z_entry (primary metric)
- Trade frequency vs. z_entry (number of trades per year)
- Win rate vs. z_entry (% of profitable trades)
- Average trade P&L vs. z_entry (mean profit per trade)
- Average holding time vs. z_entry
- Maximum drawdown vs. z_entry
- Bootstrap confidence intervals for Sharpe ratio at each threshold

**Expected Outcome**: Confirm H₁ (optimal threshold exists, likely around 2.0 - 2.5)

**Implications**:
- Higher threshold: fewer trades, higher win rate (stronger signals), lower frequency, potential underutilization of capital
- Lower threshold: more trades, lower win rate (noisier signals), higher frequency, higher transaction costs
- Tradeoff between signal quality and opportunity cost
- Optimal threshold may vary by volatility regime or pair characteristics
- If H₀ cannot be rejected: strategy may be robust to threshold choice, or Sharpe ratio may be poor metric (consider alternative objectives like Sortino, Calmar)

#### Hypothesis 3: Transaction Cost Sensitivity

**Null Hypothesis (H₀)**: Strategy remains profitable across a wide range of transaction cost assumptions (10-30 bps per leg).

**Alternative Hypothesis (H₁)**: Strategy is highly sensitive to transaction costs, with profitability disappearing above 25 bps per leg.

**Test Method**:
- Run backtests with transaction_cost_bps ∈ {5, 10, 15, 20, 25, 30}
- For each cost level, calculate complete performance metrics
- Identify breakeven cost level where Sharpe ratio = 0 (or returns = 0)
- Plot performance degradation curves
- Analyze which trade characteristics (holding time, z-score spread) are most affected by costs

**Metrics**:
- Sharpe ratio vs. transaction cost
- Annualized return vs. transaction cost
- Breakeven transaction cost (where returns = 0 or Sharpe = 0)
- Percentage of profitable trades vs. cost
- Profit factor (gross profit / gross loss) vs. cost
- Cost as % of gross P&L
- Sensitivity coefficient: ΔSharpe / Δcost

**Expected Outcome**: Confirm H₁ (costs are critical at high frequency)

**Implications**:
- Need highly accurate cost estimates for realistic performance projection
- Broker selection and negotiation crucial (sub-20 bps requires institutional access or high volume)
- May need to reduce trade frequency (higher z-score thresholds) if costs are elevated
- Execution quality (VWAP, TWAP algorithms, pairs algorithms) matters significantly
- If breakeven cost < 15 bps: strategy has limited practical viability without sophisticated execution
- If breakeven cost > 25 bps: strategy has cushion for retail implementation with modern zero-commission brokers
- Cost sensitivity likely higher at shorter timeframes (1-min) vs. longer (daily)

#### Hypothesis 4: In-Sample vs. Out-of-Sample Performance Degradation

**Null Hypothesis (H₀)**: In-sample optimized parameters perform equally well out-of-sample.

**Alternative Hypothesis (H₁)**: In-sample optimization leads to overfitting, with significant degradation in out-of-sample performance.

**Test Method**:
- Split data: 60% in-sample (IS), 40% out-of-sample (OOS)
- Optimize parameters on in-sample data (grid search for z_entry, zscore_window, stop loss)
- Test optimized parameters on held-out out-of-sample data
- Compare IS vs. OOS performance metrics
- Calculate degradation percentage = (Sharpe_IS - Sharpe_OOS) / Sharpe_IS
- Bootstrap confidence intervals for IS and OOS Sharpe ratios
- Test parameter stability: correlation between IS-optimal and OOS-optimal parameters

**Metrics**:
- Sharpe ratio: in-sample vs. out-of-sample
- Annualized return: IS vs. OOS
- Maximum drawdown: IS vs. OOS
- Win rate: IS vs. OOS
- Degradation percentage (for Sharpe, returns, win rate)
- Parameter stability: rank correlation of parameter performance IS vs. OOS
- Statistical significance of degradation (paired t-test on rolling window Sharpes)

**Expected Outcome**: Confirm H₁ (some degradation expected, but < 30%)

**Implications**:
- Expect 20-30% Sharpe degradation OOS as reasonable benchmark (per academic literature)
- Degradation > 50%: severe overfitting, parameters not robust
- Need conservative parameter selection: prefer robust over optimal
- Walk-forward analysis critical for realistic performance estimation
- If H₀ confirmed (no degradation): either parameters truly robust or insufficient difference between IS and OOS periods (regime stability)
- Result informs confidence in forward performance projections

### Phase 2: Robustness Testing

#### Hypothesis 5: Stop Loss Effectiveness for Tail Risk Reduction

**Null Hypothesis (H₀)**: A stop loss at z = 4.0 does not improve risk-adjusted returns compared to no stop loss.

**Alternative Hypothesis (H₁)**: The stop loss significantly reduces tail risk and improves Sharpe ratio by preventing catastrophic losses from cointegration breakdown.

**Test Method**:
- Compare backtest results with three configurations:
  1. With stop loss (z_stop = 4.0) - baseline
  2. Without stop loss (z_stop = ∞) - let positions run
  3. Alternative stop levels (z_stop ∈ {3.0, 3.5, 4.5, 5.0}) - sensitivity analysis
- Analyze drawdown and tail risk metrics
- Calculate percentage of trades stopped out vs. mean-reverted
- Analyze P&L distribution: compare left tail (5th percentile) with and without stops
- Test during crisis periods specifically (identify regime breaks)

**Metrics**:
- Maximum drawdown (with vs. without stop loss)
- 95th percentile loss (Value at Risk)
- Conditional Value at Risk (CVaR, expected loss beyond VaR)
- Sharpe ratio (risk-adjusted returns)
- Sortino ratio (downside deviation)
- Percentage of trades stopped out vs. mean-reverted
- Average P&L of stopped trades vs. mean-reverted trades
- Tail ratio (95th percentile gain / 5th percentile loss)

**Expected Outcome**: Confirm H₁ (stop loss improves risk-adjusted returns significantly)

**Implications**:
- Stop loss critical for risk management, prevents catastrophic losses from permanent relationship breakdowns
- Stopped-out trades indicate pair quality deterioration; should trigger pair re-evaluation
- Optimal stop level balances false positives (premature exit before reversion) vs. false negatives (letting losers run)
- Tighter stops (z = 3.0): lower drawdown but higher false positive rate, may reduce profitability
- Wider stops (z = 5.0): capture more reversions but allow larger drawdowns
- If H₀ confirmed: either pairs are very stable (rare stops triggered) or stop level is suboptimal (too wide or too tight)
- Result informs whether to implement pair-specific stops based on volatility or half-life

#### Hypothesis 6: Walk-Forward Robustness

**Null Hypothesis (H₀)**: Strategy performance is unstable across rolling walk-forward windows (high variance in out-of-sample Sharpe ratios).

**Alternative Hypothesis (H₁)**: Strategy delivers consistent risk-adjusted returns across multiple walk-forward periods, indicating genuine alpha rather than data mining.

**Test Method**:
- Use anchored or rolling walk-forward methodology:
  - 6-month in-sample period for parameter optimization
  - 3-month out-of-sample holdout for testing
  - Roll forward by 3 months, repeat
- For each OOS period, calculate Sharpe ratio using IS-optimized parameters
- Test consistency: percentage of periods with Sharpe > 1.0 (or > 0)
- Calculate mean and standard deviation of OOS Sharpe ratios
- Plot OOS Sharpe over time to identify regime changes
- Compare anchored vs. rolling walk-forward (does recency help?)

**Metrics**:
- Mean out-of-sample Sharpe ratio across all periods
- Standard deviation of OOS Sharpe ratios (lower = more consistent)
- Percentage of OOS periods with positive Sharpe
- Percentage of OOS periods with Sharpe > 1.0
- Worst OOS period Sharpe (downside risk)
- Information ratio (mean OOS Sharpe / std dev OOS Sharpe)
- Time series plot of OOS Sharpe ratios

**Expected Outcome**: Uncertain (critical robustness test)

**Implications**:
- If H₁ confirmed (consistent performance): strategy has genuine edge, not overfit to specific period
- If H₀ confirmed (inconsistent): strategy is curve-fit to historical data, parameters lack predictive power
- Low variance in OOS Sharpe: robust strategy across regimes
- High variance: performance highly regime-dependent, need regime filters or adaptive parameters
- Percentage of positive periods should be > 60% for viable strategy
- Negative periods should cluster (regime changes) rather than scatter randomly
- Result is most important validation: determines confidence in forward deployment

#### Hypothesis 7: P-Value as Pair Quality Signal

**Null Hypothesis (H₀)**: The p-value from the ADF cointegration test does not predict future pair profitability.

**Alternative Hypothesis (H₁)**: Pairs with lower p-values (stronger statistical significance of cointegration) generate higher risk-adjusted returns in backtesting.

**Test Method**:
- Sort all cointegrated pairs into quintiles by ADF test p-value (Q1 = lowest p-values, strongest cointegration)
- Run separate backtests for each quintile portfolio
- Compare Sharpe ratios across quintiles
- Analyze other metrics (win rate, average P&L, cointegration persistence) by quintile
- Statistical test: regression of pair Sharpe on p-value (or rank)

**Metrics**:
- Sharpe ratio by p-value quintile (expect decreasing with higher p-value)
- Average trade P&L by quintile
- Win rate by quintile
- Half-life by quintile (expect shorter for lower p-values)
- Percentage of pairs maintaining cointegration OOS by quintile
- Regression coefficient: Sharpe vs. p-value

**Expected Outcome**: Confirm H₁ (lower p-value → better performance)

**Implications**:
- Justifies focusing on top-ranked pairs (lowest p-values)
- Informs max_pairs selection: quality over quantity
- P-value threshold of 0.05 may be too lenient; consider 0.01 or 0.001 for stricter filtering
- If H₁ confirmed: can use p-value for pair weighting (allocate more capital to low p-value pairs)
- If H₀ confirmed: p-value is purely statistical artifact, doesn't predict trading profitability; need alternative quality metrics (half-life, R², variance ratio test)
- Result informs pair ranking and selection algorithm

---

## Literature Review

This literature review covers the theoretical foundations and empirical evidence supporting cointegration-based pairs trading strategies. Papers are organized thematically to provide context for the research hypotheses and methodology employed in this project.

### Foundational Theory

#### Engle, R. F., & Granger, C. W. J. (1987)

**Citation**: Engle, R. F., & Granger, C. W. J. (1987). "Co-integration and Error Correction: Representation, Estimation, and Testing." *Econometrica*, 55(2), 251-276.

**Thesis**: This seminal paper introduces cointegration theory, demonstrating that two non-stationary I(1) time series can possess a stationary I(0) linear combination, implying a long-run equilibrium relationship exploitable for econometric modeling and forecasting.

**Main Points and Findings**:

The paper establishes the formal definition of cointegration: two integrated series $X_t$ and $Y_t$ are cointegrated of order (d,b) if both are I(d) but a linear combination $Z_t = Y_t - \beta X_t$ is I(d-b) with b > 0, where typically d=1 and b=1, yielding a stationary spread from two non-stationary price series. The Engle-Granger two-step estimation procedure first estimates the long-run relationship via ordinary least squares regression ($Y_t = \alpha + \beta X_t + \epsilon_t$), then tests the residuals for stationarity using the Augmented Dickey-Fuller (ADF) test with modified critical values to account for the fact that $\beta$ is estimated rather than known. The Error Correction Model (ECM) representation demonstrates that cointegrated variables must have an error correction form, where short-run dynamics adjust to deviations from long-run equilibrium: $\Delta Y_t = \gamma(Y_{t-1} - \beta X_{t-1}) + \text{other terms}$, providing the theoretical foundation for mean reversion in trading strategies. The paper proves the Granger Representation Theorem, establishing that cointegration and error correction are equivalent representations of the same underlying dynamic system, giving researchers flexibility in modeling approach.

**Relevance to Current Research**:

This paper provides the core statistical methodology for our pair selection process, as we employ the Engle-Granger two-step method to identify cointegrated equity pairs and the ADF test to validate spread stationarity at the 95% confidence level (p ≤ 0.05). The theoretical guarantee that cointegrated series exhibit mean reversion (via the ECM representation) directly justifies our trading strategy of entering positions when spreads deviate from equilibrium and exiting when they revert to the mean. Our Hypothesis 1 (within-sector cointegration) and Hypothesis 2 (half-life stability) are direct applications of testing whether the cointegration framework holds robustly for equity pairs, and our entire risk management framework depends on the premise that statistically validated cointegration relationships will persist through the trading period.

#### Elliott, R. J., Van Der Hoek, J., & Malcolm, W. P. (2005)

**Citation**: Elliott, R. J., Van Der Hoek, J., & Malcolm, W. P. (2005). "Pairs Trading." *Quantitative Finance*, 5(3), 271-276.

**Thesis**: This paper formalizes pairs trading within the framework of continuous-time stochastic processes, demonstrating that optimal entry and exit thresholds can be derived analytically when spreads follow an Ornstein-Uhlenbeck mean-reverting process.

**Main Points and Findings**:

The spread between cointegrated stocks is modeled as an Ornstein-Uhlenbeck (OU) process: $dX_t = \kappa(\mu - X_t)dt + \sigma dW_t$, where $\kappa$ is the mean-reversion speed, $\mu$ is the long-run mean, $\sigma$ is volatility, and $W_t$ is a standard Wiener process, capturing the key properties of stationarity and mean reversion inherent in cointegrated relationships. The paper derives closed-form solutions for expected profit per trade as a function of entry threshold, exit threshold, and OU process parameters, demonstrating that profit maximization involves a tradeoff between waiting for extreme deviations (higher signal quality) and trade frequency (capital efficiency). Optimal thresholds are shown to depend on the mean-reversion speed $\kappa$ (related to half-life via $t_{1/2} = \ln(2)/\kappa$): faster mean reversion justifies tighter thresholds while slower reversion requires wider thresholds to ensure profitability net of transaction costs. The authors prove that for realistic parameter ranges, symmetric thresholds (±a for entry, 0 for exit) are near-optimal, supporting the industry-standard z-score approach.

**Relevance to Current Research**:

This paper provides the theoretical foundation for our z-score threshold selection (Hypothesis 5), as the OU framework predicts that optimal entry thresholds exist and should be calibrated to mean-reversion speed, which we proxy through half-life estimation. Our default z-score entry threshold of ±2.5 represents approximately 2.5 standard deviations, consistent with the OU-derived optimal thresholds for typical equity pair parameters (κ ≈ 0.1-0.5 day⁻¹, σ ≈ 0.01-0.05), and our hypothesis testing framework explicitly examines whether empirical backtests confirm the theoretical prediction of threshold optimality. The linkage between $\kappa$ and half-life directly motivates our half-life filter (5-120 days), as the OU model becomes less reliable for extreme mean-reversion speeds, and the symmetric threshold structure (long spread at z < -2.5, short spread at z > +2.5) matches the OU-optimal strategy.

### Pairs Trading Strategies

#### Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006)

**Citation**: Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). "Pairs Trading: Performance of a Relative-Value Arbitrage Rule." *Review of Financial Studies*, 19(3), 797-827.

**Thesis**: This landmark empirical study demonstrates that pairs trading based on the distance method generated significant excess returns averaging 11% annually with Sharpe ratios around 2.0 during 1962-2002, though profitability has declined post-1990, likely due to increased competition and market efficiency.

**Main Points and Findings**:

Using a comprehensive dataset of US equities over four decades (1962-2002), the authors implement a distance-based pairs trading strategy that matches stocks by minimizing the sum of squared deviations between normalized price series over a 12-month formation period, then trades the top 20 pairs during a 6-month trading period, repeating this process in rolling windows. The strategy generates average annualized excess returns of approximately 11% before transaction costs, with Sharpe ratios consistently around 2.0 in the early decades, significantly outperforming a matched sample of random long positions, demonstrating genuine arbitrage rather than compensation for systematic risk factors. Profitability exhibits clear time-series patterns: strongest in the 1970s and 1980s, declining markedly after 1990, with Sharpe ratios falling to around 1.0 in the 1990s, consistent with the "limits to arbitrage" hypothesis that successful strategies attract capital and competition, ultimately eroding returns. Transaction cost sensitivity analysis reveals that the strategy remains profitable at reasonable cost assumptions (25-50 basis points round-trip) but profitability is significantly reduced at higher cost levels, highlighting the importance of execution quality and suggesting that post-2000 deterioration may partly reflect tightening profit margins rather than complete strategy failure.

**Relevance to Current Research**:

This paper establishes the historical benchmark for pairs trading performance, allowing us to calibrate our expected return targets (realistic Sharpe ratio of 1.5-2.0) and providing context for interpreting our backtest results, particularly given that we are testing in the post-2000 period when Gatev et al. documented alpha decay. The finding that profitability declined over time directly motivates our hypothesis testing framework, especially Hypothesis 16 (out-of-sample degradation) and Hypothesis 17 (walk-forward robustness), as we must demonstrate that our cointegration-based approach remains viable in the modern, more efficient market environment rather than relying on historical profitability that may no longer exist. The transaction cost sensitivity results inform our conservative 20 basis point per leg cost assumption (Hypothesis 12) and emphasize that our strategy must achieve positive risk-adjusted returns even with realistic trading costs to be practically implementable, and the alpha decay pattern warns us to expect lower returns than historical studies and to build in robustness through continuous monitoring and parameter adaptation.

#### Do, B., & Faff, R. (2010)

**Citation**: Do, B., & Faff, R. (2010). "Does Simple Pairs Trading Still Work?" *Financial Analysts Journal*, 66(4), 83-95.

**Thesis**: This paper examines whether pairs trading strategies remain profitable in modern markets (1963-2009), comparing distance and cointegration methods and finding that while both methods were profitable pre-2000, the cointegration approach demonstrates superior robustness post-2000, suggesting it captures more fundamental economic relationships.

**Main Points and Findings**:

Testing on US equities from 1963-2009, the authors implement both the traditional distance method (Gatev et al. 2006) and a cointegration-based method using Engle-Granger tests, controlling for transaction costs, short-sale constraints, and market microstructure effects to ensure fair comparison. Both methods generated statistically significant profits during 1963-1999 with Sharpe ratios of 1.5-2.5 and annualized returns of 8-12%, confirming the Gatev et al. findings and extending them to include cointegration methodology, but the critical divergence occurs post-2000: distance method profitability essentially disappears (Sharpe < 0.5, often statistically insignificant), while cointegration method maintains modest profitability (Sharpe 0.8-1.2, statistically significant in most test periods). The superior post-2000 performance of cointegration is attributed to its theoretical foundation in long-run equilibrium relationships rather than purely statistical price similarity: cointegration tests explicitly verify spread stationarity (required for mean reversion) whereas distance matching may identify spurious relationships that break down out-of-sample. Transaction cost analysis reveals that cointegration pairs tend to have longer half-lives, resulting in longer holding periods and fewer trades, which makes the strategy more robust to transaction costs despite potentially lower trade frequency, whereas distance pairs often mean-revert quickly but require more frequent rebalancing.

**Relevance to Current Research**:

This paper provides crucial validation for our choice of cointegration methodology over alternative pair selection approaches, as the finding that cointegration-based pairs trading maintains profitability in modern markets (post-2000) while distance methods fail directly supports our research approach and suggests our 2023-2024 backtest period may still exhibit exploitable opportunities. The paper's emphasis on the theoretical superiority of cointegration (testing for stationarity rather than assuming it) reinforces our Hypothesis 1 (within-sector cointegration) and Hypothesis 3 (p-value as quality signal), as the cointegration framework provides formal statistical tests that should predict pair quality and profitability. The finding that transaction costs matter greatly and that cointegration pairs benefit from longer holding periods informs our parameter selection for maximum holding time (Hypothesis 8) and z-score thresholds (Hypothesis 5), suggesting we should optimize for trade quality over trade quantity to minimize cost drag, and the overall message that "simple" strategies face increasing challenges in efficient markets motivates our comprehensive hypothesis testing to identify which refinements (stop losses, time limits, portfolio diversification) add genuine value versus overfitting.

#### Do, B., & Faff, R. (2012)

**Citation**: Do, B., & Faff, R. (2012). "Are Pairs Trading Profits Robust to Trading Costs?" *Journal of Financial Research*, 35(2), 261-287.

**Thesis**: This paper rigorously examines the impact of transaction costs on pairs trading profitability, demonstrating that strategies are highly sensitive to cost assumptions, with breakeven costs typically in the 15-20 basis points per leg range, making execution quality critical for practical implementation.

**Main Points and Findings**:

Using US equity data and both distance and cointegration methodologies, the authors systematically vary transaction cost assumptions from 5 to 50 basis points per leg (10-100 bps round-trip) to map the cost-profitability frontier, finding that gross Sharpe ratios of 2.0-2.5 decline linearly with costs at approximately 0.05 Sharpe units per additional basis point. Breakeven analysis reveals critical thresholds: at 15 bps per leg (60 bps round-trip), most pairs trading strategies remain profitable but with sharply reduced returns; at 20 bps per leg (80 bps round-trip, our baseline assumption), only the highest-quality pairs and most disciplined execution maintain positive risk-adjusted returns; above 25 bps per leg, profitability disappears for most implementations. High-frequency strategies face particularly acute cost sensitivity: intraday rebalancing or tight stop losses that increase turnover can easily push effective costs above breakeven levels even with nominally low commission rates, as bid-ask spreads and market impact dominate commissions for frequent traders. The paper documents that cost sensitivity varies by pair characteristics: longer half-life pairs with fewer trades per year tolerate higher costs better than fast-mean-reverting pairs, and larger-cap stocks with tighter spreads allow more aggressive trading than mid-cap stocks, suggesting pair selection and universe design should explicitly consider transaction cost regimes.

**Relevance to Current Research**:

This paper provides the empirical foundation for our Hypothesis 12 (transaction cost sensitivity), as we must explicitly test whether our strategy remains profitable under conservative cost assumptions (20 bps per leg) and identify the breakeven cost level to assess practical viability and margin of safety. Our intraday implementation (1-minute bars) faces maximum cost pressure according to Do & Faff's findings, making realistic cost modeling essential rather than optional, and our hypothesis testing must verify that gross alpha is sufficient to overcome the 80 basis point round-trip cost hurdle. The finding that pair characteristics influence cost tolerance directly informs our pair selection methodology: we should prefer pairs with longer half-lives (Hypothesis 2) and higher cointegration significance (Hypothesis 3) as these generate fewer, higher-quality trades that better withstand transaction costs. The paper's emphasis on execution quality motivates potential future extensions including smart order routing, VWAP algorithms, or pairs-specific execution algorithms to minimize market impact, and the overall message that costs can eliminate statistical arbitrage profits entirely reinforces that our strategy's success depends as much on operational excellence (broker selection, execution technology) as on statistical sophistication.

### Cointegration Methods

#### Avellaneda, M., & Lee, J. H. (2010)

**Citation**: Avellaneda, M., & Lee, J. H. (2010). "Statistical Arbitrage in the U.S. Equities Market." *Quantitative Finance*, 10(7), 761-782.

**Thesis**: This paper demonstrates that Principal Component Analysis (PCA) can identify mean-reverting eigenvectors in multi-stock portfolios, enabling a more sophisticated statistical arbitrage approach than pairwise cointegration, with empirical results showing Sharpe ratios around 3.0 before costs on daily US equity data.

**Main Points and Findings**:

Rather than testing pairwise cointegration, the authors construct the covariance matrix of returns for a universe of stocks (e.g., all stocks in a sector) and perform eigendecomposition to identify principal components representing systematic factors and mean-reverting residuals, trading deviations from the statistical equilibrium defined by the PCA model. The methodology involves (1) estimating the covariance matrix over a formation period, (2) identifying mean-reverting eigenvectors through their eigenvalues (small eigenvalues correspond to slow-moving, mean-reverting factors), (3) constructing market-neutral portfolios along these eigenvectors, and (4) entering trades when portfolio deviations exceed threshold levels measured in standard deviations. Empirical tests on S&P 500 constituents (2000-2008) demonstrate gross Sharpe ratios of approximately 3.0 on daily data, significantly higher than pairwise cointegration approaches, attributed to better utilization of information by exploiting multi-stock relationships rather than restricting to pairs, though the strategy requires more sophisticated risk management including eigenportfolio rebalancing and factor exposure monitoring. The approach naturally handles more than two stocks and can identify sector-wide or factor-based arbitrage opportunities invisible to pairwise methods, but comes with increased complexity in portfolio construction, execution (simultaneous trades in multiple stocks), and computational requirements (large covariance matrix estimation and eigendecomposition).

**Relevance to Current Research**:

This paper represents an advanced extension beyond pairwise cointegration that we may pursue in future work (Phase 4: Advanced Extensions), as the PCA-based approach can potentially improve Sharpe ratios and reduce strategy capacity constraints by trading baskets rather than pairs, while leveraging the same fundamental concept of mean reversion in stationary combinations. The significantly higher Sharpe ratios (3.0 vs. 1.5-2.0 for pairs) suggest potential performance gains, though likely requiring more sophisticated infrastructure including multi-leg execution algorithms and portfolio optimization capabilities beyond our current scope. The PCA methodology provides context for understanding our strategy's position in the statistical arbitrage landscape: our pairwise approach is simpler and more transparent but potentially leaves alpha on the table by ignoring multi-stock relationships, and the sector-based universe construction we employ (within-sector pairs) could naturally extend to sector-level PCA models in future iterations. The paper also validates the general mean-reversion framework in modern markets (tests through 2008, including financial crisis), providing evidence that statistical arbitrage remains viable despite increased competition, though the post-crisis period (2008-present) requires further validation which our 2023-2024 backtest will provide.

#### Caldeira, J., & Moura, G. V. (2013)

**Citation**: Caldeira, J., & Moura, G. V. (2013). "Selection of a Portfolio of Pairs Based on Cointegration: A Statistical Arbitrage Strategy." *Brazilian Review of Finance*, 11(1), 49-80.

**Thesis**: This paper systematically compares multiple cointegration testing methodologies for pairs selection (Engle-Granger, Johansen, distance-based) applied to portfolio construction, finding that the Engle-Granger approach is robust and appropriate for pairwise trading, while the Johansen test offers advantages for multi-stock baskets.

**Main Points and Findings**:

The authors test three pair/portfolio selection methodologies on Brazilian equity markets: (1) Engle-Granger two-step for pairwise cointegration (our approach), (2) Johansen maximum likelihood test capable of detecting multiple cointegrating vectors among more than two stocks, and (3) distance method (sum of squared deviations) as a non-cointegration benchmark. For pairwise trading (two stocks), Engle-Granger proves robust with statistical power comparable to Johansen while being computationally simpler, validating its widespread use in practitioner implementations; the test correctly identifies cointegrated pairs with Type I error rates near the nominal 5% level and reasonable power against alternatives. For triplets and larger baskets (3-5 stocks), Johansen test demonstrates superior performance as it can detect multiple cointegrating relationships simultaneously (e.g., three stocks with two independent cointegrating vectors), whereas applying Engle-Granger pairwise to all combinations may miss complex multi-stock relationships or produce redundant trades. Distance method underperforms both cointegration approaches in out-of-sample profitability and drawdown metrics, confirming Do & Faff (2010) findings that statistical verification of stationarity (via ADF/Johansen tests) is superior to purely descriptive distance measures. Portfolio construction using cointegration-filtered pairs shows improved diversification and risk-adjusted returns compared to unfiltered universes, with optimal portfolio sizes of 10-15 pairs balancing diversification benefits against pair quality dilution.

**Relevance to Current Research**:

This paper validates our methodological choice of Engle-Granger two-step testing for pairwise cointegration, as Caldeira & Moura demonstrate it is the appropriate tool for our scope (trading pairs rather than larger baskets) with good statistical properties and computational efficiency suitable for testing hundreds or thousands of potential pairs. The comparison with Johansen test provides context for potential future extensions: our current pairwise framework could be generalized to trade triplets or sector baskets using Johansen methodology, potentially improving diversification and capturing more complex arbitrage relationships beyond two-stock pairs. The finding that optimal portfolio size is 10-15 pairs directly informs our Hypothesis 14 (portfolio diversification), suggesting our default max_pairs = 10 is theoretically sound and that marginal benefits of additional pairs likely diminish beyond this level. The paper's emphasis on out-of-sample validation and walk-forward testing aligns with our robustness testing framework (Hypotheses 16, 17, 18), reinforcing that in-sample cointegration detection must be validated through genuine out-of-sample trading to distinguish statistical significance from economic profitability, and the Brazilian market results (an emerging market with potentially different microstructure) suggest cointegration-based pairs trading is robust across market regimes, though our US equity tests on 2023-2024 data will provide more directly relevant evidence.

#### Alexander, C., & Dimitriu, A. (2005)

**Citation**: Alexander, C., & Dimitriu, A. (2005). "Indexing and Statistical Arbitrage." *Journal of Portfolio Management*, 31(2), 50-63.

**Thesis**: This paper argues that cointegration can be applied to both index tracking (replicating benchmark returns with fewer securities) and statistical arbitrage (exploiting mean reversion), demonstrating that factor models combined with cointegration testing provide a unified framework for constructing market-neutral portfolios with improved risk-adjusted returns.

**Main Points and Findings**:

The authors present a two-stage methodology: first, apply PCA to identify common factors (systematic risk components) driving returns in a universe of stocks; second, test residuals from the factor model for cointegration to identify mean-reverting arbitrage opportunities orthogonal to systematic factors, ensuring market neutrality. For index tracking, cointegration-based stock selection identifies securities that replicate the index through long-run relationships rather than short-term correlation, resulting in tracking portfolios that maintain fidelity through regime changes and require less frequent rebalancing than traditional optimization approaches. For statistical arbitrage, the factor model decomposition isolates idiosyncratic risk from systematic risk, allowing cointegration tests to focus on true pair-specific relationships rather than spurious correlation induced by common factor exposures; this improves pair quality and reduces false positives in cointegration testing. Out-of-sample validation on European equities (1995-2002) demonstrates that cointegration-screened portfolios maintain Sharpe ratios of 1.5-2.0 in test periods, whereas portfolios based purely on in-sample optimization degrade significantly, confirming the importance of statistical testing for stationarity rather than relying on sample moments. The paper emphasizes the necessity of walk-forward testing with rolling parameter estimation: hedge ratios and factor loadings must be re-estimated periodically (quarterly or semi-annually) to adapt to structural changes, with the cointegration framework providing formal tests to detect relationship breakdown.

**Relevance to Current Research**:

This paper reinforces the theoretical foundation for market-neutral statistical arbitrage through cointegration while highlighting the potential enhancement from incorporating factor models, which we have not yet implemented but could pursue in advanced extensions (residualizing pair returns by Fama-French factors before cointegration testing). The emphasis on market neutrality through factor decomposition validates our dollar-neutral position construction (Major Point 3 in précis), though Alexander & Dimitriu suggest we could improve true beta-neutrality by explicitly controlling for systematic factor exposures beyond simple dollar matching. The strong out-of-sample performance (Sharpe 1.5-2.0 maintained in holdout periods) provides an aspirational benchmark for our Hypothesis 16 (in-sample vs. out-of-sample) and Hypothesis 17 (walk-forward robustness), suggesting that properly implemented cointegration strategies can maintain consistent performance if parameter re-estimation and pair re-testing are performed systematically. The recommendation for periodic hedge ratio re-estimation (quarterly/semi-annually) informs our potential implementation of Hypothesis 21 (Kalman filter for dynamic hedge ratios), as Alexander & Dimitriu's results suggest static hedge ratios may underperform even without the complexity of Kalman filtering, and the application to European markets provides evidence of global applicability for cointegration-based stat arb, though our US equity focus with modern data (2023-2024) will provide the most relevant validation for current market conditions.